# HtmlHeaderSplitter
### 分割h1-h6标签里面的文本

### HTMLHeaderTextSplitter：基于标题的标准切分器
```
• 它是怎么工作的？ 它会在文本的元素级别进行扫描，一旦遇到你指定的标题标签（如 <h1>, <h2>），就会把前面的内容切成一个块。
• 元数据注入： 它最强大的地方在于，切出来的每一个 Document 块，都会在其 Metadata 中层层带上它所属的父级标题信息
  （例如：{'Header 1': '第一章', 'Header 2': '第一节'}）。
• 缺点： 如果你的 HTML 页面没有规范的 <h1>~<h6> 标签（很多现代网页使用 <span class="title">），它就会失效，直接退化成一个大块。
  此外，它会洗掉所有的 HTML 标签。
```

In [1]:
from langchain_text_splitters import HTMLHeaderTextSplitter

# 2.定义HTML文件
html_string = """
<!DOCTYPE html>
<html>
<body>
    <div>
        <h1>欢迎来到尚硅谷！</h1>
        <p>尚硅谷是专门培训IT技术方向</p>
        <div>
            <h2>尚硅谷老师简介</h2>
            <p>尚硅谷老师拥有多年教学经验，都是从一线互联网下来</p>
            <h3>尚硅谷北京校区</h3>
            <p>北京校区位于宏福科技园区</p>
        </div>
    </div>
</body>
</html>"""

# 3.用于指定要根据哪些HTML标签来分割文本
headers_to_split_on = [
    ("h1", "标题1"),
    ("h2", "标题2"),
    ("h3", "标题3"),
]

splitter = HTMLHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
)

result = splitter.split_text(html_string)
print(result)

[Document(metadata={'标题1': '欢迎来到尚硅谷！'}, page_content='欢迎来到尚硅谷！'), Document(metadata={'标题1': '欢迎来到尚硅谷！'}, page_content='尚硅谷是专门培训IT技术方向'), Document(metadata={'标题1': '欢迎来到尚硅谷！', '标题2': '尚硅谷老师简介'}, page_content='尚硅谷老师简介'), Document(metadata={'标题1': '欢迎来到尚硅谷！', '标题2': '尚硅谷老师简介'}, page_content='尚硅谷老师拥有多年教学经验，都是从一线互联网下来'), Document(metadata={'标题1': '欢迎来到尚硅谷！', '标题2': '尚硅谷老师简介', '标题3': '尚硅谷北京校区'}, page_content='尚硅谷北京校区'), Document(metadata={'标题1': '欢迎来到尚硅谷！', '标题2': '尚硅谷老师简介', '标题3': '尚硅谷北京校区'}, page_content='北京校区位于宏福科技园区')]


#  HTMLSectionSplitter
```

2. HTMLSectionSplitter：基于视觉/逻辑章节的智能切分器

• 它是怎么工作的？ 它是 HTMLHeaderTextSplitter 的升级版，不仅看标准标题，还通过 XSLT 转换技术 去探测网页中的“视觉章节”。
• 特色功能： 如果一个网页的开发者没有用 <h1>，而是通过调整 <span style="font-size: 24px"> 来做标题，HTMLSectionSplitter
  可以通过配置，智能地把这种基于字体大小的元素识别为章节边界。
• 适用场景： 特别适合处理结构不规范的新闻、博客文章或爬虫直接抓取的杂乱网页
```

In [3]:
from langchain_text_splitters import HTMLSectionSplitter

# 2.定义HTML文件
html_string = """
<!DOCTYPE html>
<html>
<body>
    <div>
        <h1>欢迎来到尚硅谷！</h1>
        <p>尚硅谷是专门培训IT技术方向</p>
        <div>
            <h2>尚硅谷老师简介</h2>
            <p>尚硅谷老师拥有多年教学经验，都是从一线互联网下来</p>
            <h3>尚硅谷北京校区</h3>
            <p>北京校区位于宏福科技园区</p>
        </div>
    </div>
</body>
</html>"""

# 3.用于指定要根据哪些HTML标签来分割文本
headers_to_split_on = [
    ("h1", "标题1"),
    ("h2", "标题2"),
    ("h3", "标题3"),
]

splitter = HTMLSectionSplitter(
    headers_to_split_on=headers_to_split_on,
)

docs = splitter.split_text(html_string)
for doc in docs:
    print(doc.page_content)


欢迎来到尚硅谷！ 
 尚硅谷是专门培训IT技术方向
尚硅谷老师简介 
 尚硅谷老师拥有多年教学经验，都是从一线互联网下来
尚硅谷北京校区 
 北京校区位于宏福科技园区


# HTMLSemanticPreservingSplitter
```
 HTMLSemanticPreservingSplitter：最先进的语义保留切分器

• 它是怎么工作的？ 这是目前功能最全面的 HTML 切分器。它在按照标题切分的同时，首要任务是保护 HTML 的复杂语义不被破坏。
• 王牌优势（保留复杂结构）：
	• 表格与列表保护：如果网页中包含复杂的 <table> 或 <ul>，普通的切分器可能会在表格中间强行切断，导致 LLM 无法理解数据。
      它会尽量保证整个表格或列表在同一个 Chunk 里。
	• 媒体和多模态友好：它支持将网页中的 <a>（超链接）、<img>（图片）、<video> 标签，直接转换为类似 Markdown 的格式（如 [文本](链接)、
      ![alt](图片地址)），从而把多模态信息带给大模型。
• 大小控制： 提供了 max_chunk_size 和 chunk_overlap 属性。如果一个章节太长，它会使用 RecursiveCharacterTextSplitter
   进行二次切分，但在切分时依然会尽量避开表格等核心元素
```

In [4]:
from langchain_text_splitters import HTMLSemanticPreservingSplitter

# 2.定义HTML文件
html_string = """
<!DOCTYPE html>
<html>
<body>
    <div>
        <h1>欢迎来到尚硅谷！</h1>
        <p>尚硅谷是专门培训IT技术方向</p>
        <div>
            <h2>尚硅谷老师简介</h2>
            <p>尚硅谷老师拥有多年教学经验，都是从一线互联网下来</p>
            <h3>尚硅谷北京校区</h3>
            <p>北京校区位于宏福科技园区</p>
        </div>
    </div>
</body>
</html>"""

# 3.用于指定要根据哪些HTML标签来分割文本
headers_to_split_on = [
    ("h1", "标题1"),
    ("h2", "标题2"),
    ("h3", "标题3"),
]

splitter = HTMLSemanticPreservingSplitter(
    headers_to_split_on=headers_to_split_on,
)

docs = splitter.split_text(html_string)
for doc in docs:
    print(doc.page_content)


尚硅谷是专门培训IT技术方向
尚硅谷老师拥有多年教学经验，都是从一线互联网下来
北京校区位于宏福科技园区


C:\Users\kenny\AppData\Local\Temp\ipykernel_996\3243212668.py:28: LangChainBetaWarning: The class `HTMLSemanticPreservingSplitter` is in beta. It is actively being worked on, so the API may change.
  splitter = HTMLSemanticPreservingSplitter(
